# Lecture 7 — Selecting the latent rank $k$ and regularization $\lambda$

The rank $k$ and regularization strength $\lambda$ are hyperparameters: $k$ controls the number of latent dimensions, while $\lambda$ controls the strength of the penalty on large latent factors. This experiment fits several candidate combinations of $(k,\lambda)$ on the same observed entries and evaluates both observed and held-out reconstruction error.

Because this is a synthetic matrix, the complete matrix is known and the hidden entries can be evaluated against the truth. Here those hidden entries play the role of held-out validation ratings. In a real recommender system, the analogous comparison would use a validation split of known ratings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matrix_factorization import factorize, reconstruct

np.set_printoptions(precision=3, suppress=True)

## 1. Select $k$ and $\lambda$ using held-out RMSE

We compare several values of $k$ and several values of $\lambda$. We use a logarithmic grid for $\lambda$ because useful regularization strengths can differ by orders of magnitude.

For each pair $(k,\lambda)$, we train the factorization model and calculate RMSE only on the hidden entries. In this synthetic experiment, the hidden entries are the held-out validation set, so their values come from $X_{\text{true}}$:

$$
RMSE_{\mathrm{val}}(k,\lambda)=\sqrt{\frac{1}{|\Omega_{\mathrm{val}}|}\sum_{(a,i)\in\Omega_{\mathrm{val}}}(X_{\text{true},ai}-u_a^Tv_i)^2}.
$$

The selected pair is the combination with the lowest validation RMSE:

$$
(k^*,\lambda^*)=\arg\min_{k,\lambda}RMSE_{\mathrm{val}}(k,\lambda).
$$

In [ ]:
# The true matrix has rank 2. We hide roughly 45% of its entries.
rng = np.random.default_rng(42)
n_users, n_movies, k_true = 8, 7, 2
U_true = rng.normal(size=(n_users, k_true))
V_true = rng.normal(size=(n_movies, k_true))
X_true = U_true @ V_true.T

mask = rng.random((n_users, n_movies)) < 0.55
Y_synth = np.where(mask, X_true, np.nan)

candidate_k = range(1, 6)
candidate_lambda = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
results = []

for k in candidate_k:
    for lam in candidate_lambda:
        U_hat, V_hat, history = factorize(Y_synth, k=k, lambda_=lam, seed=1)
        X_hat = reconstruct(U_hat, V_hat)

        observed_rmse = np.sqrt(np.mean((X_hat[mask] - X_true[mask])**2))
        validation_rmse = np.sqrt(np.mean((X_hat[~mask] - X_true[~mask])**2))
        results.append({
            'k': k,
            'lambda': lam,
            'observed_rmse': observed_rmse,
            'validation_rmse': validation_rmse,
        })

results = pd.DataFrame(results)
results = results.sort_values(['k', 'lambda'])
print(results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_row = results.loc[results['validation_rmse'].idxmin()]
best_k = int(best_row['k'])
best_lambda = float(best_row['lambda'])

print(f'\nSelected k*: {best_k}')
print(f'Selected lambda*: {best_lambda:g}')
print(f'Best validation RMSE: {best_row["validation_rmse"]:.4f}')

## 2. Inspect the validation scores

The plot shows how validation RMSE changes with $\lambda$ for each candidate rank. The best model is the point with the lowest validation RMSE.

A very small $\lambda$ allows the factors more freedom to fit the observed data, while a large $\lambda$ penalizes large factor values more strongly. Validation performance helps us choose a useful balance.

In [ ]:
fig, ax = plt.subplots()

for k in candidate_k:
    subset = results[results['k'] == k]
    ax.plot(subset['lambda'], subset['validation_rmse'], marker='o', label=f'k={k}')

ax.set_xscale('log')
ax.set_xlabel('Regularization strength lambda')
ax.set_ylabel('Validation RMSE')
ax.set_title('Validation RMSE for candidate (k, lambda) pairs')
ax.legend()
plt.show()

## Interpretation

The important quantity is the **held-out/validation RMSE**. Increasing $k$ gives the model more flexibility, so the observed error can decrease. But lower observed error does not necessarily mean better predictions for unseen entries.

In this synthetic experiment the true matrix was generated with rank $2$. The held-out RMSE can favor the lower-dimensional model, while larger ranks may fit the observed entries slightly better without improving the held-out predictions. This illustrates why $k$ should be selected using validation performance rather than by choosing the largest rank.

The same validation principle is used for $\lambda$. Very weak regularization may allow the factors to become large, while very strong regularization can restrict the model too much. The validation RMSE helps identify a useful balance.

For a real recommender system, the missing entries are not available as ground truth. Instead, known ratings would be split into training and validation sets, and both $k$ and $\lambda$ would be selected by the validation score.

## What to remember

- $k$ is the number of latent dimensions. It is a hyperparameter, not a learned rating or an entry of $U$ or $V$.
- $\lambda$ controls the strength of regularization. It is also a hyperparameter.
- Candidate values of $\lambda$ are usually searched on a logarithmic scale.
- $k$ and $\lambda$ can interact, so the final model should select them by comparing candidate pairs $(k,\lambda)$.
- The selected pair is $(k^*,\lambda^*)=\arg\min_{k,\lambda}RMSE_{\mathrm{val}}(k,\lambda)$.
- After model selection, the final notebook can refit the model using the selected hyperparameters and then predict genuinely missing ratings.
- In a real recommender system, validation ratings are known ratings temporarily held out for model selection; genuinely missing ratings have no known target value.